# 04. Build regional Elo and international calibration

Add three chronological states: local Elo inside domestic pools, each team's earlier domestic home, and pool offsets learned from eligible cross-pool results. Read `matches_clean.csv` and export classified, local, and calibrated histories plus comparison reports.

Run notebooks 01-03 first because final comparisons reuse notebook 03's saved test matches. Classification currently gives each league label one role across all years; year-dependent roles need review. Exact team-name identities and potentially stale home assignments remain limitations. Unknown homes remain missing, and final-state summaries must never be joined backward onto history.


## 01. Locate the cleaned history

Find the project root and require `matches_clean.csv`. Import pandas and NumPy for classification and chronological updates. Feature exports remain under `data/processed/`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

base = Path.cwd()

if not (base / "data" / "processed").is_dir():
    base = base.parent

input_path = base / "data" / "processed" / "matches_clean.csv"
output_dir = base / "data" / "processed"

if not input_path.is_file():
    raise FileNotFoundError(
        f"Missing {input_path}. Run notebook 01 first."
    )

## 02. Inspect competition labels and coverage

Check required fields and unique IDs, parse UTC dates, and summarize game counts and date ranges by league. This inventory supports explicit classification. Source labels and dates are not independent proof of official competition identities.


In [2]:
matches = pd.read_csv(
    input_path,
    dtype={
        "match_id": "string",
        "league": "string",
        "team_a": "string",
        "team_b": "string",
    },
)

required_columns = {
    "match_id",
    "date",
    "league",
    "team_a",
    "team_b",
    "result_a",
}

missing_columns = required_columns - set(matches.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if matches[list(required_columns)].isna().any().any():
    raise ValueError("Required fields contain missing values.")

if not matches["match_id"].is_unique:
    raise ValueError("Duplicate match IDs found.")

matches["date"] = pd.to_datetime(
    matches["date"],
    utc=True,
    errors="raise",
    format="mixed",
)

competition_summary = (
    matches.groupby("league")
    .agg(
        games=("match_id", "size"),
        first_game=("date", "min"),
        last_game=("date", "max"),
    )
    .sort_values("games", ascending=False)
    .reset_index()
)

print("Competitions:", len(competition_summary))
display(competition_summary)

Competitions: 124


,league,games,first_game,last_game
0,LDL,7019,2018-03-03 06:15:12+00:00,2024-06-05 11:24:34+00:00
1,LPL,6918,2017-01-19 09:45:22+00:00,2026-09-13 12:03:56+00:00
2,LCK,4978,2017-01-17 07:10:23+00:00,2026-09-13 08:19:05+00:00
3,LCKC,2664,2021-01-18 07:53:00+00:00,2026-09-15 09:47:51+00:00
4,VCS,2526,2018-01-18 08:41:54+00:00,2026-10-11 12:15:18+00:00
...,...,...,...,...
119,MSC,25,2020-05-28 04:36:28+00:00,2020-05-31 11:41:52+00:00
120,Riot,20,2020-05-27 08:46:58+00:00,2020-05-30 12:37:45+00:00
121,AC,19,2026-03-04 21:12:25+00:00,2026-03-08 20:51:01+00:00
122,LGL,8,2019-04-06 11:43:13+00:00,2019-04-07 13:18:53+00:00


## 03. Define domestic rating pools

Map supported domestic labels to distinct local-rating pools. EU LCS to LEC and NA LCS to LCS are explicit continuity assumptions. Other listed histories remain separate, and developmental competitions do not automatically inherit a parent pool.


In [3]:
DOMESTIC_POOLS = {
    "LCK": "LCK",
    "LPL": "LPL",

    # Explicit continuity assumptions for renamed competitions.
    "LEC": "LEC",
    "EU LCS": "LEC",
    "LCS": "LCS",
    "NA LCS": "LCS",

    "PCS": "PCS",
    "VCS": "VCS",
    "CBLOL": "CBLOL",
    "LLA": "LLA",
    "LJL": "LJL",
    "TCL": "TCL",
    "LCO": "LCO",

    # Keep these histories separate initially.
    "LMS": "LMS",
    "OPL": "OPL",
    "LCL": "LCL",
    "LCP": "LCP",
    "LTA N": "LTA_N",
    "LTA S": "LTA_S",
}

## 04. Define international and other competition roles

Use MSI, WLDs, and FST as the initial global international scope. Treat LTA as a bridge connecting existing homes without creating a domestic home. Listed developmental competitions do not learn regional state; unlisted labels remain for review. Require role sets not to overlap.


In [4]:
INTERNATIONAL_COMPETITIONS = {
    "MSI",
    "WLDs",
    "FST",
}

# Regional bridge events: useful for connecting rating pools,
# but participation must not establish a new domestic home.
CROSS_POOL_COMPETITIONS = {
    "LTA",
}

# Excluded from regional learning in this first version.
# These are not assigned to their parent top-level rating pools.
DEVELOPMENTAL_COMPETITIONS = {
    "LDL",
    "LCKC",
    "CK",
    "LSPL",
    "LCSA",
    "NACL",
    "CBLOLA",
    "LJLA",
    "EU CS",
    "NA CS",
    "LJLCS",
    "OCS",
    "BRCC",
}

groups = {
    "domestic": set(DOMESTIC_POOLS),
    "international": INTERNATIONAL_COMPETITIONS,
    "cross_pool": CROSS_POOL_COMPETITIONS,
    "developmental": DEVELOPMENTAL_COMPETITIONS,
}

group_names = list(groups)

for index, first in enumerate(group_names):
    for second in group_names[index + 1:]:
        overlap = groups[first] & groups[second]

        if overlap:
            raise ValueError(
                f"Overlapping classifications: "
                f"{first}, {second}: {overlap}"
            )

## 05. Create metadata and eligibility flags

`competition_type` describes a role and `rating_pool` identifies a domestic pool. `updates_home_pool` and `use_local_elo` govern domestic state, `use_region_calibration` permits offset learning, and `use_international_model` identifies modeling scope. Start with review status and false flags, then apply explicit mappings. The flags have different purposes and need not select identical games.


In [5]:
classification = competition_summary.copy()

classification["competition_type"] = "review"

classification["rating_pool"] = pd.Series(
    pd.NA,
    index=classification.index,
    dtype="string",
)

classification["updates_home_pool"] = False
classification["use_local_elo"] = False
classification["use_region_calibration"] = False
classification["use_international_model"] = False

classification["classification_note"] = (
    "Not reviewed; excluded from regional learning."
)

# Domestic competitions.
domestic_mask = classification["league"].isin(DOMESTIC_POOLS)

classification.loc[
    domestic_mask, "competition_type"
] = "domestic"

classification.loc[
    domestic_mask, "rating_pool"
] = classification.loc[
    domestic_mask, "league"
].map(DOMESTIC_POOLS)

classification.loc[
    domestic_mask, "updates_home_pool"
] = True

classification.loc[
    domestic_mask, "use_local_elo"
] = True

classification.loc[
    domestic_mask, "classification_note"
] = "Included domestic rating pool."

# Global international events.
international_mask = classification["league"].isin(
    INTERNATIONAL_COMPETITIONS
)

classification.loc[
    international_mask, "competition_type"
] = "international"

classification.loc[
    international_mask, "use_region_calibration"
] = True

classification.loc[
    international_mask, "use_international_model"
] = True

classification.loc[
    international_mask, "classification_note"
] = "Global international event; does not update home pool."

# Cross-pool regional events.
cross_pool_mask = classification["league"].isin(
    CROSS_POOL_COMPETITIONS
)

classification.loc[
    cross_pool_mask, "competition_type"
] = "cross_pool"

classification.loc[
    cross_pool_mask, "use_region_calibration"
] = True

classification.loc[
    cross_pool_mask, "classification_note"
] = (
    "Connects historical home pools; excluded from the "
    "global international model in this version."
)

# Developmental competitions.
developmental_mask = classification["league"].isin(
    DEVELOPMENTAL_COMPETITIONS
)

classification.loc[
    developmental_mask, "competition_type"
] = "developmental"

classification.loc[
    developmental_mask, "classification_note"
] = (
    "Retained, but excluded from regional learning "
    "in this version."
)

display(classification)

,league,games,first_game,last_game,competition_type,rating_pool,updates_home_pool,use_local_elo,use_region_calibration,use_international_model,classification_note
0,LDL,7019,2018-03-03 06:15:12+00:00,2024-06-05 11:24:34+00:00,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
1,LPL,6918,2017-01-19 09:45:22+00:00,2026-09-13 12:03:56+00:00,domestic,LPL,True,True,False,False,Included domestic rating pool.
2,LCK,4978,2017-01-17 07:10:23+00:00,2026-09-13 08:19:05+00:00,domestic,LCK,True,True,False,False,Included domestic rating pool.
3,LCKC,2664,2021-01-18 07:53:00+00:00,2026-09-15 09:47:51+00:00,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
4,VCS,2526,2018-01-18 08:41:54+00:00,2026-10-11 12:15:18+00:00,domestic,VCS,True,True,False,False,Included domestic rating pool.
...,...,...,...,...,...,...,...,...,...,...,...
119,MSC,25,2020-05-28 04:36:28+00:00,2020-05-31 11:41:52+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.
120,Riot,20,2020-05-27 08:46:58+00:00,2020-05-30 12:37:45+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.
121,AC,19,2026-03-04 21:12:25+00:00,2026-03-08 20:51:01+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.
122,LGL,8,2019-04-06 11:43:13+00:00,2019-04-07 13:18:53+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.


## 06. Validate classification consistency

Require one classification per league and complete boolean flags. Home and local-rating updates need a valid domestic pool; international, bridge, developmental, and review games must not establish domestic membership. These checks enforce the mapping without establishing its historical correctness for every year.


In [6]:
assert classification["league"].is_unique

boolean_columns = [
    "updates_home_pool",
    "use_local_elo",
    "use_region_calibration",
    "use_international_model",
]

assert classification[boolean_columns].notna().all().all()

home_updates = classification["updates_home_pool"]

assert classification.loc[
    home_updates, "rating_pool"
].notna().all()

assert classification.loc[
    home_updates, "competition_type"
].eq("domestic").all()

assert classification.loc[
    classification["use_local_elo"],
    "rating_pool",
].notna().all()

assert not classification.loc[
    classification["competition_type"].isin(
        ["international", "cross_pool", "developmental", "review"]
    ),
    "updates_home_pool",
].any()

print("Classification checks passed.")

Classification checks passed.


## 07. Preview the basic metadata join

Attach competition type and domestic rating pool with a many-to-one join. Domestic games must have a pool; international events must not claim a new domestic pool. This initial preview is replaced by the complete flag-bearing join below.


In [7]:
classified_matches = matches.merge(
    classification[
        ["league", "competition_type", "rating_pool"]
    ],
    on="league",
    how="left",
    validate="many_to_one",
)

assert len(classified_matches) == len(matches)
assert classified_matches["competition_type"].notna().all()

domestic_matches = classified_matches.loc[
    classified_matches["competition_type"].eq("domestic")
]

assert domestic_matches["rating_pool"].notna().all()

international_matches = classified_matches.loc[
    classified_matches["competition_type"].eq("international")
]

assert international_matches["rating_pool"].isna().all()

display(classified_matches.head(10))

,match_id,date,league,team_a,team_b,result_a,competition_type,rating_pool
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0,developmental,<NA>
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0,developmental,<NA>
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0,developmental,<NA>
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1,developmental,<NA>
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0,developmental,<NA>
5,2017-01-11 15:33:10+00:00|EU CS|TREU/1001490139,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,1,developmental,<NA>
6,2017-01-11 15:46:32+00:00|EU CS|TREU/1001490141,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,0,developmental,<NA>
7,2017-01-11 16:44:20+00:00|EU CS|TREU/1001490143,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,0,developmental,<NA>
8,2017-01-11 17:53:03+00:00|EU CS|TREU/1001490147,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,0,developmental,<NA>
9,2017-01-11 18:59:33+00:00|EU CS|TREU/1001490149,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,0,developmental,<NA>


## 08. Measure classification coverage

Summarize games by role, list domestic pools, and count games eligible for each learning component. These counts describe the consequences of the chosen mapping. They are coverage statistics, not model performance.


In [8]:
coverage = (
    classification.groupby("competition_type")
    .agg(
        competitions=("league", "size"),
        games=("games", "sum"),
    )
)

coverage["game_share"] = (
    coverage["games"] / coverage["games"].sum()
)

display(coverage)

print("Domestic rating pools:")
display(
    classification.loc[
        classification["competition_type"].eq("domestic"),
        ["league", "rating_pool", "games"],
    ]
)

print("Games eligible for each learning component:")

for flag in boolean_columns:
    game_count = classification.loc[ classification[flag], "games"].sum()
    print(f"{flag}: {game_count:,}")

,competitions,games,game_share
competition_type,,,
cross_pool,1,40,0.000416
developmental,13,17307,0.180141
domestic,19,33616,0.349893
international,3,1952,0.020317
review,88,43160,0.449232


Domestic rating pools:


,league,rating_pool,games
1,LPL,LPL,6918
2,LCK,LCK,4978
4,VCS,VCS,2526
5,LEC,LEC,2267
6,TCL,TCL,2258
7,LJL,LJL,2206
9,CBLOL,CBLOL,2005
11,LCS,LCS,1918
14,PCS,PCS,1786
25,LLA,LLA,1153


Games eligible for each learning component:
updates_home_pool: 33,616
use_local_elo: 33,616
use_region_calibration: 1,992
use_international_model: 1,952


## 09. Attach complete competition metadata

Join every eligibility flag and classification note while preserving match count and uniqueness. Sort by timestamp and match ID. Review games remain available in the history even though they cannot update regional state.


In [9]:
mapping_columns = [
    "league",
    "competition_type",
    "rating_pool",
    *boolean_columns,
    "classification_note",
]

classified_matches = matches.merge(
    classification[mapping_columns],
    on="league",
    how="left",
    validate="many_to_one",
)

assert len(classified_matches) == len(matches)
assert classified_matches["match_id"].is_unique
assert classified_matches["competition_type"].notna().all()

classified_matches = (
    classified_matches
    .sort_values(["date", "match_id"])
    .reset_index(drop=True)
)

display(classified_matches.head(10))

,match_id,date,league,team_a,team_b,result_a,competition_type,rating_pool,updates_home_pool,use_local_elo,use_region_calibration,use_international_model,classification_note
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
5,2017-01-11 15:33:10+00:00|EU CS|TREU/1001490139,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,1,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
6,2017-01-11 15:46:32+00:00|EU CS|TREU/1001490141,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
7,2017-01-11 16:44:20+00:00|EU CS|TREU/1001490143,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
8,2017-01-11 17:53:03+00:00|EU CS|TREU/1001490147,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
9,2017-01-11 18:59:33+00:00|EU CS|TREU/1001490149,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."


## 10. Export the classification audit

Save competition metadata, classified matches, and competitions awaiting review. Review status excludes games from regional learning rather than deleting them from global Elo. Rerunning replaces these processed exports.


In [10]:
output_dir.mkdir(parents=True, exist_ok=True)

classification.to_csv(
    output_dir / "competition_classification.csv",
    index=False,
)

classified_matches.to_csv(
    output_dir / "matches_classified.csv",
    index=False,
)

classification.loc[
    classification["competition_type"].eq("review")
].to_csv(
    output_dir / "competitions_to_review.csv",
    index=False,
)

print("Saved classification and classified matches.")

print(
    "Unreviewed games retained:",
    classified_matches["competition_type"].eq("review").sum(),
)

Saved classification and classified matches.
Unreviewed games retained: 43160


## 11. Set local Elo parameters and check inputs

New `(rating_pool, team)` pairs start at 1500 with K=24 and scale=400. Require regional columns, unique IDs, timestamps, binary results, and boolean flags. These are fixed baseline settings, not settings chosen from test performance.


In [11]:
INITIAL_LOCAL_ELO = 1500.0
LOCAL_K = 24.0
LOCAL_ELO_SCALE = 400.0

required_regional_columns = [
    "match_id",
    "date",
    "league",
    "team_a",
    "team_b",
    "result_a",
    "competition_type",
    "rating_pool",
    "updates_home_pool",
    "use_local_elo",
    "use_region_calibration",
    "use_international_model",
]

missing_columns = (
    set(required_regional_columns)
    - set(classified_matches.columns)
)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if classified_matches.empty:
    raise ValueError("No classified matches available.")

if not classified_matches["match_id"].is_unique:
    raise ValueError("Duplicate match IDs found.")

if classified_matches["date"].isna().any():
    raise ValueError("Missing timestamps found.")

if not classified_matches["result_a"].isin([0, 1]).all():
    raise ValueError("Results must be 0 or 1.")

for column in boolean_columns:
    if not classified_matches[column].isin([True, False]).all():
        raise ValueError(
            f"{column} must contain actual boolean values."
        )

print("Regional input checks passed.")

Regional input checks passed.


## 12. Define the local Elo probability

Apply the standard Elo formula to local ratings and clip extreme exponents for numerical stability. Check equal-rating probability and opponent symmetry. Local ratings measure strength within their pool; separate 1500 starting points cannot establish relative pool strength.


In [12]:
def local_expected_score(rating_a, rating_b):
    exponent = np.clip(
        (rating_b - rating_a) / LOCAL_ELO_SCALE,
        -300,
        300,
    )

    return 1.0 / (1.0 + 10.0 ** exponent)


assert np.isclose(local_expected_score(1500, 1500), 0.5)

assert np.isclose(
    local_expected_score(1600, 1500)
    + local_expected_score(1500, 1600),
    1.0,
)

print("Local Elo probability checks passed.")

Local Elo probability checks passed.


## 13. Generate local ratings and historical homes

Domestic participation identifies the applicable pool for that match. Other games use only membership established at an earlier timestamp, and simultaneous conflicting domestic assignments are rejected. Read frozen ratings and homes for each batch, then apply summed domestic changes, counts, and home updates. International games do not update local Elo, and unknown homes keep local ratings missing.


In [13]:
def generate_regional_features(
    match_table,
    initial_elo=INITIAL_LOCAL_ELO,
    k=LOCAL_K,
):
    local_ratings = {}
    local_game_counts = {}
    historical_homes = {}

    records = []

    ordered = match_table.sort_values(["date", "match_id"])

    for timestamp, batch in ordered.groupby("date", sort=True):
        pending_rating_changes = {}
        pending_game_counts = {}
        pending_home_updates = {}

        for row in batch.itertuples(index=False):
            # Domestic participation identifies the pool for this match.
            # Otherwise, use only previously established membership.
            if row.updates_home_pool:
                if pd.isna(row.rating_pool):
                    raise ValueError(
                        f"Missing domestic pool for {row.match_id}"
                    )

                pool_a = row.rating_pool
                pool_b = row.rating_pool

                for team in [row.team_a, row.team_b]:
                    previous_assignment = pending_home_updates.get(team)

                    if (
                        previous_assignment is not None
                        and previous_assignment != row.rating_pool
                    ):
                        raise ValueError(
                            f"{team} appears in different domestic pools "
                            f"at the same timestamp: {timestamp}"
                        )

                    pending_home_updates[team] = row.rating_pool

            else:
                pool_a = historical_homes.get(row.team_a)
                pool_b = historical_homes.get(row.team_b)

            known_a = pool_a is not None
            known_b = pool_b is not None

            key_a = (pool_a, row.team_a) if known_a else None
            key_b = (pool_b, row.team_b) if known_b else None

            pre_local_a = (
                local_ratings.get(key_a, initial_elo)
                if known_a
                else np.nan
            )

            pre_local_b = (
                local_ratings.get(key_b, initial_elo)
                if known_b
                else np.nan
            )

            games_a = (
                local_game_counts.get(key_a, 0)
                if known_a
                else 0
            )

            games_b = (
                local_game_counts.get(key_b, 0)
                if known_b
                else 0
            )

            record = row._asdict()

            record.update(
                {
                    "home_pool_a": pool_a,
                    "home_pool_b": pool_b,
                    "home_known_a": known_a,
                    "home_known_b": known_b,
                    "pre_local_elo_a": pre_local_a,
                    "pre_local_elo_b": pre_local_b,
                    "local_elo_diff": pre_local_a - pre_local_b,
                    "pre_local_games_a": games_a,
                    "pre_local_games_b": games_b,
                }
            )

            records.append(record)

            # Only eligible domestic matches change local Elo.
            if row.use_local_elo:
                if (
                    not row.updates_home_pool
                    or not known_a
                    or not known_b
                    or pool_a != pool_b
                ):
                    raise ValueError(
                        f"Invalid local-Elo configuration: {row.match_id}"
                    )

                probability_a = local_expected_score(
                    pre_local_a,
                    pre_local_b,
                )

                delta = k * (row.result_a - probability_a)

                pending_rating_changes[key_a] = (
                    pending_rating_changes.get(key_a, 0.0)
                    + delta
                )

                pending_rating_changes[key_b] = (
                    pending_rating_changes.get(key_b, 0.0)
                    - delta
                )

                for key in [key_a, key_b]:
                    pending_game_counts[key] = (
                        pending_game_counts.get(key, 0) + 1
                    )

        # Apply results after every prediction at this timestamp.
        for key, change in pending_rating_changes.items():
            local_ratings[key] = (
                local_ratings.get(key, initial_elo) + change
            )

        for key, count in pending_game_counts.items():
            local_game_counts[key] = (
                local_game_counts.get(key, 0) + count
            )

        historical_homes.update(pending_home_updates)

    return (
        pd.DataFrame(records),
        local_ratings,
        local_game_counts,
        historical_homes,
    )

## 14. Test home chronology with synthetic fixtures

Use constructed teams and pools to test initial local ratings, later access to earlier domestic results, and missing membership before a domestic appearance. A domestic appearance at the same timestamp cannot establish an international home for that batch. These fixtures are test input, not historical match records.


In [14]:
example = pd.DataFrame(
    {
        "match_id": ["example_0", "example_1", "example_2", "example_3"],
        "date": pd.to_datetime(
            [
                "2020-01-01",
                "2020-01-01",
                "2020-01-02",
                "2020-01-03",
            ],
            utc=True,
        ),
        "league": ["WLDs", "Example R1", "Example R2", "WLDs"],
        "team_a": ["A", "A", "C", "A"],
        "team_b": ["C", "B", "D", "C"],
        "result_a": [1, 1, 0, 1],
        "competition_type": [
            "international",
            "domestic",
            "domestic",
            "international",
        ],
        "rating_pool": [None, "R1", "R2", None],
        "updates_home_pool": [False, True, True, False],
        "use_local_elo": [False, True, True, False],
        "use_region_calibration": [True, False, False, True],
        "use_international_model": [True, False, False, True],
    }
)

example_features, example_ratings, _, _ = (
    generate_regional_features(example)
)

example_features = example_features.set_index("match_id")

# No historical domestic membership exists at the first timestamp.
assert not example_features.loc["example_0", "home_known_a"]
assert not example_features.loc["example_0", "home_known_b"]

# First domestic appearances start at 1500.
assert np.isclose(
    example_features.loc["example_1", "pre_local_elo_a"],
    1500,
)

# Later international features use earlier domestic results.
assert example_features.loc["example_3", "home_pool_a"] == "R1"
assert example_features.loc["example_3", "home_pool_b"] == "R2"

assert np.isclose(
    example_features.loc["example_3", "pre_local_elo_a"],
    1512,
)

assert np.isclose(
    example_features.loc["example_3", "pre_local_elo_b"],
    1488,
)

# International results did not update local Elo.
assert np.isclose(example_ratings[("R1", "A")], 1512)
assert np.isclose(example_ratings[("R2", "C")], 1488)

# A future domestic result cannot alter an earlier prediction.
changed_example = example.copy()
changed_example.loc[
    changed_example["match_id"].eq("example_1"),
    "result_a",
] = 0

changed_features, _, _, _ = generate_regional_features(changed_example)
changed_features = changed_features.set_index("match_id")

assert not changed_features.loc["example_0", "home_known_a"]

assert pd.isna(
    changed_features.loc["example_0", "pre_local_elo_a"]
)

print("Regional chronology checks passed.")

Regional chronology checks passed.


## 15. Replay regional state for every game

Generate prior homes, local ratings, differences, and domestic game counts for the complete classified history. Check row preservation, chronological ordering, and complete domestic features. Returned dictionaries describe final state only.


In [15]:
(
    regional_features,
    final_local_ratings,
    final_local_game_counts,
    final_home_pools,
) = generate_regional_features(classified_matches)

assert len(regional_features) == len(classified_matches)
assert regional_features["match_id"].is_unique
assert regional_features["date"].is_monotonic_increasing

domestic_rows = regional_features.loc[
    regional_features["use_local_elo"]
]

assert domestic_rows[
    ["pre_local_elo_a", "pre_local_elo_b", "local_elo_diff"]
].notna().all().all()

assert domestic_rows["home_pool_a"].eq(
    domestic_rows["home_pool_b"]
).all()

print("Regional feature rows:", len(regional_features))

display(
    regional_features[
        [
            "date",
            "league",
            "team_a",
            "team_b",
            "home_pool_a",
            "home_pool_b",
            "pre_local_elo_a",
            "pre_local_elo_b",
            "local_elo_diff",
        ]
    ].head(10)
)

Regional feature rows: 96075


,date,league,team_a,team_b,home_pool_a,home_pool_b,pre_local_elo_a,pre_local_elo_b,local_elo_diff
0,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,NaN,NaN,NaN,NaN,NaN
1,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,NaN,NaN,NaN,NaN,NaN
2,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,NaN,NaN,NaN,NaN,NaN
3,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,NaN,NaN,NaN,NaN,NaN
4,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,NaN,NaN,NaN,NaN,NaN
5,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,NaN,NaN,NaN,NaN,NaN
6,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,NaN,NaN,NaN,NaN,NaN
7,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,NaN,NaN,NaN,NaN,NaN
8,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,NaN,NaN,NaN,NaN,NaN
9,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,NaN,NaN,NaN,NaN,NaN


## 16. Report international home coverage

Count games with both homes known for each international-model competition, then inspect unknown-home examples. Unknown membership is retained for fallback and is never inferred from later appearances. Coverage measures feature availability, not prediction quality.


In [16]:
international_rows = regional_features.loc[
    regional_features["use_international_model"]
].copy()

international_rows["both_homes_known"] = (
    international_rows["home_known_a"]
    & international_rows["home_known_b"]
)

home_coverage = (
    international_rows.groupby("league")
    .agg(
        games=("match_id", "size"),
        both_homes_known=("both_homes_known", "sum"),
    )
)

home_coverage["known_home_share"] = (
    home_coverage["both_homes_known"] / home_coverage["games"]
)

display(home_coverage)

unknown_home_matches = international_rows.loc[
    ~international_rows["both_homes_known"],
    [
        "date",
        "league",
        "team_a",
        "team_b",
        "home_pool_a",
        "home_pool_b",
    ],
]

print("International games with an unknown home:", len(unknown_home_matches))
display(unknown_home_matches.head(20))

,games,both_homes_known,known_home_share
league,,,
FST,80,80,1.000000
MSI,696,647,0.929598
WLDs,1176,1133,0.963435


International games with an unknown home: 92


,date,league,team_a,team_b,home_pool_a,home_pool_b
2552,2017-04-29 17:55:15+00:00,MSI,Lyon Gaming,Virtus.pro,NaN,LCL
2553,2017-04-29 19:26:17+00:00,MSI,GAM Esports,Isurus,NaN,NaN
2554,2017-04-29 20:33:35+00:00,MSI,Isurus,Virtus.pro,NaN,LCL
2555,2017-04-29 21:30:33+00:00,MSI,GAM Esports,Lyon Gaming,NaN,NaN
2556,2017-04-29 22:46:58+00:00,MSI,Isurus,Lyon Gaming,NaN,NaN
2557,2017-04-29 23:39:36+00:00,MSI,GAM Esports,Virtus.pro,NaN,LCL
2564,2017-05-01 18:00:05+00:00,MSI,Isurus,Virtus.pro,NaN,LCL
2565,2017-05-01 19:09:45+00:00,MSI,GAM Esports,Lyon Gaming,NaN,NaN
2566,2017-05-01 19:59:53+00:00,MSI,GAM Esports,Isurus,NaN,NaN
2567,2017-05-01 20:51:31+00:00,MSI,Lyon Gaming,Virtus.pro,NaN,LCL


## 17. Check pool conservation and summarize local ratings

Build final local ratings and domestic appearance counts. Verify that each pool's total equals 1500 times its number of rated teams. Raw local rankings describe relative performance inside a pool, so comparing different pools requires calibration.


In [17]:
local_rating_rows = [
    {
        "rating_pool": pool,
        "team": team,
        "local_elo": rating,
        "domestic_games": final_local_game_counts.get((pool, team), 0),
    }
    for (pool, team), rating in final_local_ratings.items()
]

local_ratings_table = pd.DataFrame(
    local_rating_rows,
    columns=["rating_pool", "team", "local_elo", "domestic_games"],
)

if local_ratings_table.empty:
    raise ValueError("No local ratings were generated.")

pool_checks = (
    local_ratings_table.groupby("rating_pool")
    .agg(
        teams=("team", "size"),
        total_rating=("local_elo", "sum"),
    )
)

pool_checks["expected_total"] = (
    pool_checks["teams"] * INITIAL_LOCAL_ELO
)

np.testing.assert_allclose(
    pool_checks["total_rating"],
    pool_checks["expected_total"],
)

print("Within-pool rating conservation passed.")
display(pool_checks)

display(
    local_ratings_table.sort_values(
        ["rating_pool", "local_elo"],
        ascending=[True, False],
    ).head(30)
)

Within-pool rating conservation passed.


,teams,total_rating,expected_total
rating_pool,,,
CBLOL,35,52500.0,52500.0
LCK,30,45000.0,45000.0
LCL,22,33000.0,33000.0
LCO,15,22500.0,22500.0
LCP,15,22500.0,22500.0
LCS,26,39000.0,39000.0
LEC,30,45000.0,45000.0
LJL,51,76500.0,76500.0
LLA,17,25500.0,25500.0


,rating_pool,team,local_elo,domestic_games
299,CBLOL,LØS,1758.229646,191
225,CBLOL,FURIA,1737.275851,323
256,CBLOL,LOUD,1664.812691,292
67,CBLOL,RED Canids,1618.148501,400
259,CBLOL,Vorax Liberty,1602.587005,48
224,CBLOL,Prodigy Esports,1565.863524,47
244,CBLOL,Santos e-Sports,1538.218444,21
483,CBLOL,KaBuM! Ilha das Lendas,1537.244452,18
46,CBLOL,paiN Gaming,1535.252501,443
306,CBLOL,Vivo Keyd Stars,1529.826926,179


## 18. Export local features and final-state summaries

Save local history, final ratings, latest homes, and unknown-home audits. The latest-home summary must not be joined backward onto past matches. Historical rows already contain the home that was available at prediction time.


In [18]:
regional_features.to_csv(
    output_dir / "regional_elo_features_all_years.csv",
    index=False,
)

local_ratings_table.to_csv(
    output_dir / "local_elo_ratings_all_years.csv",
    index=False,
)

home_pool_table = pd.DataFrame(
    [
        {"team": team, "latest_home_pool": pool}
        for team, pool in final_home_pools.items()
    ],
    columns=["team", "latest_home_pool"],
)

home_pool_table.to_csv(
    output_dir / "latest_team_home_pools.csv",
    index=False,
)

unknown_home_matches.to_csv(
    output_dir / "international_unknown_home_matches.csv",
    index=False,
)

print("Regional outputs saved to:", output_dir)

Regional outputs saved to: C:\Users\USER\lol_worlds_predictor\data\processed


## 19. Check reproducibility of local features

Recompute regional features and compare ordering, homes, ratings, differences, and prior counts with the currently saved export. In a top-to-bottom run, that export was just written above, so this checks reproducibility rather than an independent historical snapshot. Classification and identity limitations remain.


In [19]:
saved_regional = pd.read_csv(output_dir / "regional_elo_features_all_years.csv")
replayed_regional, _, _, _ = generate_regional_features(classified_matches)
assert saved_regional["match_id"].tolist() == replayed_regional["match_id"].tolist()
for column in ["home_pool_a", "home_pool_b"]:
    assert saved_regional[column].fillna("").tolist() == replayed_regional[column].fillna("").tolist()
for column in ["pre_local_elo_a", "pre_local_elo_b", "local_elo_diff", "pre_local_games_a", "pre_local_games_b"]:
    np.testing.assert_allclose(saved_regional[column], replayed_regional[column], equal_nan=True)
print("Saved local features match chronological replay:", len(replayed_regional))

Saved local features match chronological replay: 96075


## 20. Learn pool offsets chronologically

Use `adjusted_strength = pre_local_elo + pre_pool_offset`, with offsets initially zero, region K=12, and scale=400. Only flagged games between distinct known pools with finite local strengths transfer `12 * (result - expected)` between offsets. All equal-timestamp games read frozen offsets and prior calibration counts before summed updates. Same-pool and unknown-home games cannot update offsets.


In [20]:
REGION_K = 12.0
REGION_SCALE = 400.0

def generate_calibrated_features(table):
    if table.empty or not table["match_id"].is_unique:
        raise ValueError("Expected nonempty, unique match history.")
    if table["date"].isna().any() or not table["result_a"].isin([0, 1]).all():
        raise ValueError("Invalid timestamps or results.")
    if not table["use_region_calibration"].isin([True, False]).all():
        raise ValueError("Calibration flags must be boolean.")
    offsets, counts, records = {}, {}, []
    for timestamp, batch in table.sort_values(["date", "match_id"]).groupby("date", sort=True):
        changes, additions = {}, {}
        for row in batch.itertuples(index=False):
            pools = [row.home_pool_a, row.home_pool_b]
            local = [row.pre_local_elo_a, row.pre_local_elo_b]
            known = [pd.notna(pool) for pool in pools]
            pre = [offsets.get(pool, 0.0) if ok else np.nan for pool, ok in zip(pools, known)]
            prior = [counts.get(pool, 0) if ok else np.nan for pool, ok in zip(pools, known)]
            adjusted = np.asarray(local, dtype=float) + pre
            valid = all(known) and np.isfinite(adjusted).all()
            diff = adjusted[0] - adjusted[1] if valid else np.nan
            probability = 1 / (1 + 10.0 ** np.clip(-diff / REGION_SCALE, -300, 300)) if valid else np.nan
            eligible = bool(row.use_region_calibration and valid and pools[0] != pools[1])
            delta = REGION_K * (row.result_a - probability) if eligible else 0.0
            record = row._asdict()
            record.update(pre_pool_offset_a=pre[0], pre_pool_offset_b=pre[1],
                          pre_pool_calibration_games_a=prior[0], pre_pool_calibration_games_b=prior[1],
                          adjusted_strength_a=adjusted[0], adjusted_strength_b=adjusted[1],
                          adjusted_strength_diff=diff, calibrated_probability_a=probability,
                          calibration_eligible=eligible, pool_delta_a=delta, pool_delta_b=-delta)
            records.append(record)
            for pool, ok in zip(pools, known):
                if ok:
                    offsets.setdefault(pool, 0.0)
                    counts.setdefault(pool, 0)
            if eligible:
                for pool, change in zip(pools, [delta, -delta]):
                    changes[pool] = changes.get(pool, 0.0) + change
                    additions[pool] = additions.get(pool, 0) + 1
        assert np.isclose(sum(changes.values()), 0.0, atol=1e-9)
        for pool, change in changes.items():
            offsets[pool] += change
            counts[pool] += additions[pool]
    return pd.DataFrame(records), offsets, counts

## 21. Test offset isolation and excluded updates

Use explicitly synthetic fixtures to check timestamp isolation, prior counts, zero-sum transfers, and independence of prior features from that batch's results. Unknown homes, same-pool games, disabled flags, and nonfinite ratings must not update offsets. Also verify rejection of conflicting simultaneous domestic assignments.


In [21]:
fixture = pd.DataFrame({
    "match_id": ["synthetic_" + str(i) for i in range(7)],
    "date": pd.to_datetime(["2000-01-01"] * 2 + ["2000-01-02"] * 5, utc=True),
    "home_pool_a": ["R1"] * 7,
    "home_pool_b": ["R2", "R2", "R2", None, "R1", "R2", "R2"],
    "pre_local_elo_a": [1500.] * 7,
    "pre_local_elo_b": [1500.] * 6 + [np.nan],
    "result_a": [1] * 7,
    "use_region_calibration": [True] * 5 + [False, True],
})
f, offsets, counts = generate_calibrated_features(fixture)
np.testing.assert_allclose(f.loc[:1, "pre_pool_offset_a"], 0)
np.testing.assert_allclose(f.loc[:1, "calibrated_probability_a"], 0.5)
assert f.loc[2, "pre_pool_offset_a"] == 12
assert f.loc[2, "pre_pool_calibration_games_a"] == 2
assert not f.loc[3:, "calibration_eligible"].any()
assert f.loc[3:, "pool_delta_a"].eq(0).all()
assert pd.isna(f.loc[3, "calibrated_probability_a"])
assert pd.isna(f.loc[6, "calibrated_probability_a"])
assert np.isclose(sum(offsets.values()), 0)
assert counts == {"R1": 3, "R2": 3}
changed = fixture.copy()
changed.loc[changed["date"].eq(changed["date"].max()), "result_a"] = 0
changed_f, _, _ = generate_calibrated_features(changed)
pre_columns = [c for c in f if c.startswith("pre_") or c.startswith("adjusted_")] + ["calibrated_probability_a"]
pd.testing.assert_frame_equal(f[pre_columns], changed_f[pre_columns])
shuffled_f, _, _ = generate_calibrated_features(fixture.iloc[::-1])
pd.testing.assert_frame_equal(f, shuffled_f)
conflict = example.copy()
conflicting_row = conflict.loc[conflict["match_id"].eq("example_1")].copy()
conflicting_row["match_id"] = "synthetic_conflict"
conflicting_row["rating_pool"] = "R2"
try:
    generate_regional_features(pd.concat([conflict, conflicting_row], ignore_index=True))
except ValueError as error:
    assert "different domestic pools" in str(error)
else:
    raise AssertionError("Conflicting domestic assignments were accepted.")
print("Calibration isolation, conservation, missing-home and exclusion checks passed.")

Calibration isolation, conservation, missing-home and exclusion checks passed.


## 22. Export calibrated features and regional summaries

Run calibration on all regional rows and verify conservation and update counts. Each eligible game contributes one count to each participating pool, so overall pool counts sum to twice the eligible-game count. Save `calibrated_elo_features_all_years.csv` and `regional_calibration_summary.csv`. Final offsets describe the end of history and cannot be joined backward; evaluation follows below.


In [22]:
calibrated_features, final_pool_offsets, final_calibration_counts = generate_calibrated_features(regional_features)
assert calibrated_features["match_id"].tolist() == regional_features["match_id"].tolist()
assert np.isclose(sum(final_pool_offsets.values()), 0.0, atol=1e-9)
assert sum(final_calibration_counts.values()) == 2 * calibrated_features["calibration_eligible"].sum()
np.testing.assert_allclose(calibrated_features["pool_delta_a"] + calibrated_features["pool_delta_b"], 0)
assert calibrated_features.loc[~calibrated_features["calibration_eligible"], "pool_delta_a"].eq(0).all()
pool_summary = pd.DataFrame([
    {"rating_pool": pool, "final_pool_offset": offset,
     "calibration_games": final_calibration_counts[pool]}
    for pool, offset in sorted(final_pool_offsets.items())
])
calibrated_features.to_csv(output_dir / "calibrated_elo_features_all_years.csv", index=False)
pool_summary.to_csv(output_dir / "regional_calibration_summary.csv", index=False)
print("Retained matches:", len(calibrated_features))
print("Eligible calibration games:", int(calibrated_features["calibration_eligible"].sum()))
print("Games with finite calibrated probabilities:", int(calibrated_features["calibrated_probability_a"].notna().sum()))
display(pool_summary)

Retained matches: 96075
Eligible calibration games: 1645
Games with finite calibrated probabilities: 36901


,rating_pool,final_pool_offset,calibration_games
0,CBLOL,-79.382319,130
1,LCK,305.601736,601
2,LCL,-105.422707,64
3,LCO,-70.197132,44
4,LCP,47.444641,104
5,LCS,180.779502,387
6,LEC,151.697115,545
7,LJL,-155.405466,82
8,LLA,-114.113533,65
9,LMS,-87.249002,116


## 23. Reuse the exact baseline test matches

Load notebook 03's saved boundaries and predictions, requiring identical IDs, dates, leagues, opponents, and outcomes. Regional predictions require known homes and finite local ratings. Full-scope comparisons explicitly use global Elo where regional predictions are unavailable. This remains online evaluation on an already-inspected development period.


In [23]:
baseline_dir = base / "results" / "baseline"
splits = pd.read_csv(baseline_dir / "chronological_splits.csv")
for column in ["start", "end"]:
    splits[column] = pd.to_datetime(splits[column], utc=True)
assert set(splits["split"]) == {"training", "validation", "testing"}
assert splits["split"].is_unique
baseline_test = pd.read_csv(baseline_dir / "test_predictions.csv")
baseline_test["date"] = pd.to_datetime(baseline_test["date"], utc=True)
assert baseline_test["match_id"].is_unique
test_split = splits.set_index("split").loc["testing"]
test_start, test_end = test_split["start"], test_split["end"]
regional_test = calibrated_features.loc[calibrated_features["date"].between(test_start, test_end)].copy()
assert len(regional_test) == int(test_split["matches"]) == len(baseline_test)
assert set(regional_test["match_id"]) == set(baseline_test["match_id"])
identity = ["match_id", "date", "league", "team_a", "team_b", "result_a"]
pd.testing.assert_frame_equal(
    regional_test[identity].sort_values("match_id").reset_index(drop=True),
    baseline_test[identity].sort_values("match_id").reset_index(drop=True),
    check_dtype=False,
)
evaluation = regional_test.merge(
    baseline_test[["match_id", "elo_probability_a"]],
    on="match_id", how="left", validate="one_to_one",
)
evaluation["regional_available"] = (
    evaluation["home_known_a"] & evaluation["home_known_b"]
    & np.isfinite(evaluation[["pre_local_elo_a", "pre_local_elo_b"]]).all(axis=1)
)
evaluation["local_probability_a"] = np.where(
    evaluation["regional_available"],
    1 / (1 + 10.0 ** np.clip(-evaluation["local_elo_diff"] / 400.0, -300, 300)),
    np.nan,
)
assert evaluation.loc[evaluation["regional_available"], "calibrated_probability_a"].notna().all()
for source, target in [
    ("local_probability_a", "local_with_global_fallback"),
    ("calibrated_probability_a", "calibrated_with_global_fallback"),
]:
    evaluation[target] = evaluation[source].where(
        evaluation["regional_available"], evaluation["elo_probability_a"]
    )
    assert evaluation[target].between(0, 1).all()
print("Exact baseline test period:", test_start, "through", test_end)
print("Identical test matches:", len(evaluation))

Exact baseline test period: 2025-02-05 18:58:03+00:00 through 2026-10-26 21:54:37+00:00
Identical test matches: 19204


## 24. Compare identical populations and report coverage

Compare global, local-only, and calibrated Elo on the common available subset and on full scope with global fallback. Report accuracy, log loss, Brier score, ROC AUC, fixed-bin calibration, and counts for all games, international scope, MSI plus WLDs, and WLDs. Higher accuracy/AUC and lower log loss/Brier are preferable. Require reproduction of global baseline metrics before interpreting regional differences.


In [24]:
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

def regional_metrics(frame, column):
    y = frame["result_a"].to_numpy(dtype=int)
    p = frame[column].to_numpy(dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    if len(y) == 0:
        return {"matches": 0, "accuracy": np.nan, "log_loss": np.nan,
                "brier": np.nan, "roc_auc": np.nan}
    return {
        "matches": len(y), "accuracy": accuracy_score(y, p >= 0.5),
        "log_loss": log_loss(y, p, labels=[0, 1]),
        "brier": brier_score_loss(y, p),
        "roc_auc": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
    }

scope_masks = {
    "all": pd.Series(True, index=evaluation.index),
    "international": evaluation["use_international_model"],
    "msi_and_worlds": evaluation["league"].isin(["MSI", "WLDs"]),
    "worlds": evaluation["league"].eq("WLDs"),
}
coverage_rows, metric_rows, calibration_rows = [], [], []
for scope, mask in scope_masks.items():
    scoped = evaluation.loc[mask]
    known = scoped["regional_available"]
    coverage_rows.append({
        "scope": scope, "total_matches": len(scoped),
        "both_homes_known": int((scoped["home_known_a"] & scoped["home_known_b"]).sum()),
        "regional_available": int(known.sum()), "fallback_matches": int((~known).sum()),
        "regional_coverage": known.mean() if len(scoped) else np.nan,
    })
    for population, group, models in [
        ("common_known_home", scoped.loc[known], {
            "global_elo": "elo_probability_a", "local_elo": "local_probability_a",
            "calibrated_elo": "calibrated_probability_a"}),
        ("full_scope", scoped, {
            "global_elo": "elo_probability_a", "local_elo": "local_with_global_fallback",
            "calibrated_elo": "calibrated_with_global_fallback"}),
    ]:
        for model, column in models.items():
            metric_rows.append({"scope": scope, "population": population, "model": model,
                                **regional_metrics(group, column)})
            bins = pd.cut(group[column], np.linspace(0, 1, 11), include_lowest=True)
            table = group.assign(probability_bin=bins).groupby("probability_bin", observed=False).agg(
                predicted_probability=(column, "mean"), observed_win_rate=("result_a", "mean"),
                matches=("result_a", "size"),
            ).reset_index()
            table = table.assign(scope=scope, population=population, model=model)
            assert table["matches"].sum() == len(group)
            calibration_rows.append(table)
coverage_report = pd.DataFrame(coverage_rows)
comparison_report = pd.DataFrame(metric_rows)
calibration_report = pd.concat(calibration_rows, ignore_index=True)
for (_, _), group in comparison_report.groupby(["scope", "population"]):
    assert group["matches"].nunique() == 1
baseline_scope_metrics = pd.read_csv(baseline_dir / "test_metrics_by_scope.csv")
for scope in ["all", "msi_and_worlds", "worlds"]:
    actual = comparison_report.loc[
        comparison_report["scope"].eq(scope) & comparison_report["population"].eq("full_scope")
        & comparison_report["model"].eq("global_elo")
    ].iloc[0]
    expected = baseline_scope_metrics.loc[
        baseline_scope_metrics["scope"].eq(scope) & baseline_scope_metrics["model"].eq("global_elo")
    ].iloc[0]
    np.testing.assert_allclose(actual[["matches", "accuracy", "log_loss", "brier", "roc_auc"]].to_numpy(float),
                               expected[["matches", "accuracy", "log_loss", "brier", "roc_auc"]].to_numpy(float))
display(coverage_report)
display(comparison_report)

,scope,total_matches,both_homes_known,regional_available,fallback_matches,regional_coverage
0,all,19204,7431,7431,11773,0.386951
1,international,390,390,390,0,1.000000
2,msi_and_worlds,310,310,310,0,1.000000
3,worlds,159,159,159,0,1.000000


,scope,population,model,matches,accuracy,log_loss,brier,roc_auc
0,all,common_known_home,global_elo,7431,0.629660,0.645343,0.226553,0.676764
1,all,common_known_home,local_elo,7431,0.630332,0.644119,0.225993,0.678398
2,all,common_known_home,calibrated_elo,7431,0.635042,0.641827,0.224674,0.684372
3,all,full_scope,global_elo,19204,0.630077,0.640541,0.224841,0.681242
4,all,full_scope,local_elo,19204,0.630337,0.640067,0.224624,0.681779
5,all,full_scope,calibrated_elo,19204,0.632160,0.639181,0.224114,0.684145
6,international,common_known_home,global_elo,390,0.610256,0.652837,0.231335,0.633202
7,international,common_known_home,local_elo,390,0.617949,0.662890,0.234699,0.634386
8,international,common_known_home,calibrated_elo,390,0.641026,0.646015,0.224785,0.691803
9,international,full_scope,global_elo,390,0.610256,0.652837,0.231335,0.633202


## 25. Save comparisons and assumptions

Export predictions, coverage, metrics, calibration bins, and metadata to `results/regional/`. Record test dates, fixed constants, fallback policy, and known limitations. Global-Elo fallback here is a coverage rule; notebook 06 separately learns a validation-selected blend using logistic models.


In [25]:
import json
results_dir = base / "results" / "regional"
results_dir.mkdir(parents=True, exist_ok=True)
evaluation.to_csv(results_dir / "test_predictions.csv", index=False)
coverage_report.to_csv(results_dir / "test_coverage.csv", index=False)
comparison_report.to_csv(results_dir / "test_metrics_by_scope.csv", index=False)
calibration_report.to_csv(results_dir / "test_calibration.csv", index=False)
metadata = {
    "test_start": test_start.isoformat(), "test_end": test_end.isoformat(),
    "test_matches": len(evaluation), "boundary_source": "results/baseline/chronological_splits.csv",
    "local_initial_elo": INITIAL_LOCAL_ELO, "local_k": LOCAL_K,
    "local_scale": LOCAL_ELO_SCALE, "region_k": REGION_K, "region_scale": REGION_SCALE,
    "offset_initial_value": 0.0,
    "common_subset": "both homes known and both pre-match local ratings finite",
    "fallback": "global Elo whenever the common-subset condition is false",
    "evaluation": "online Elo; all equal-timestamp predictions precede state updates",
    "test_status": "previously inspected; development evidence, not untouched final evaluation",
    "classification_limitation": "league-wide mapping, not year-specific roles",
    "identity_limitation": "exact team names; latest earlier domestic home may be stale",
}
(results_dir / "evaluation_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Saved regional comparison:", results_dir)

Saved regional comparison: C:\Users\USER\lol_worlds_predictor\results\regional
